# 4 — Evaluate: ours vs baseline, one scorer

Ours vs the released baseline, on the identical val images, through the identical `pycocotools` scorer. Writes `results/results.csv`, `results/tau_calibration.json`, `results/tuning_table.md` and the figures.

> Run **`1_reformat.ipynb` first** (it writes the CFD manifest to Drive). The Brackish frames live on the VM disk at `/content/data/brackish`; if this notebook lands on a fresh runtime, cell 1b re-creates them. Everything else (weights, run dirs, results) is on Drive and persists. On an A100 the package picks bfloat16 automatically; on a T4 it picks fp16 + GradScaler.

## 1 · Drive, paths, code

Mounts Drive, fixes the four paths every cell below uses, clones the branch and installs it editable. Safe to re-run: the clone is wiped and redone each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil, subprocess, pathlib
DRIVE = '/content/drive/MyDrive/frozen-trunk-detection'
for sub in ('weights', 'runs', 'results', 'results/manifest', 'results/viz'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
REPO = '/content/crop-counter'
DATA = '/content/data/brackish'
CFD  = '/content/cfd'
os.makedirs(CFD, exist_ok=True)
print(os.listdir(DRIVE))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/detection-head --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# torch/torchvision come with the Colab image. torchmetrics + termcolor are needed only so
# Meta's dinov3 hubconf imports (it pulls the segmentors); pycocotools for COCO AP; ijson to
# stream the 1.9M-record CFD metadata without json.load-ing it.
!pip install -q -e ".[detection]" ijson
# A running kernel does not re-read site-packages' .pth files, so the editable install is invisible
# to THIS process until restart (subprocess calls like `!python -m cropcounter.train` see it fine).
import sys, importlib
if '/content/crop-counter/src' not in sys.path:
    sys.path.insert(0, '/content/crop-counter/src')
importlib.invalidate_caches()
import cropcounter, torch
print('cropcounter', cropcounter.__file__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1b · Make sure the Brackish slice is on this VM

Colab gives each notebook its own runtime, so the frames fetched by `1_reformat.ipynb` are not here unless you attached this notebook to that same session. This cell re-creates the slice only if it is missing (metadata 47 MB, ~14.7k frames from the LILA GCS mirror; ~4–5 min).

In [ ]:
# Idempotent: skip if 1_reformat already populated this runtime.
have = os.path.exists(f'{DATA}/val/annotations.json') and os.path.isdir(f'{DATA}/val/images') and len(os.listdir(f'{DATA}/val/images')) > 0
if not have:
    META = f'{CFD}/community_fish_detection_dataset.json.zip'
    if not os.path.exists(META):
        !wget -q -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
    !python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources brackish_dataset --train-cap 100000 --val-cap 100000 --seed 0 --no-progress
    !python -m cropcounter.cfd fetch --subset {DATA} --max-side 1024 --workers 32 --mirror gcs --no-progress
!ls {DATA}/train/images | wc -l; ls {DATA}/val/images | wc -l

## 2 · Setup — reload what 1–3 produced

In [ ]:
# Rebuild everything 1-3 left behind: the run config, the backbone, the val GT and the
# baseline metrics. Nothing is recomputed -- this notebook only reads and scores.
import torch, pathlib
from cropcounter.train import TrainConfig, build_model

os.makedirs(f'{REPO}/weights', exist_ok=True)
dst = f'{REPO}/weights/dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth'
if not os.path.exists(dst):
    shutil.copy(f'{DRIVE}/weights/dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth', dst)

cfg = TrainConfig.from_json('examples/FishDetection/config_8ep.json')
cfg.data_root = pathlib.Path(DATA); cfg.weights_dir = pathlib.Path('weights')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gt = json.load(open(f'{DATA}/val/annotations.json'))
baseline_metrics = json.load(open(f'{DRIVE}/results/baseline_metrics.json'))
print(f"{len(gt['images'])} val images | baselines: {list(baseline_metrics)}")

## 3 · Inference FLOPs (measured, not quoted) — ours at 1024×576

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
box_model = build_model(cfg, device); box_model.eval()
x = torch.randn(1, 3, 576, 1024, device=device)
with torch.no_grad(), FlopCounterMode(display=False) as fc:
    box_model(x)
OURS_GFLOPS = round(fc.get_total_flops() / 1e9, 1)
print(f'ours (frozen ConvNeXt-B + decoder, 1024x576): {OURS_GFLOPS} GFLOPs')
for name, m in baseline_metrics.items():
    print(f"{name}: {m.get('gflops', 'not measured -- see 3_inference')} GFLOPs")

## 4 · Tuning — τ calibration and NMS

`tau` is the decode confidence threshold: heatmap peaks scoring below it never become predictions. Every P/R/F1 the 8-epoch run printed used the config's reference `tau` 0.3, which was picked before the run and never tuned. Calibrate it here, the same way the wheat example does — one forward pass over val, decoded **once** at `ap_tau`, then every `tau` in 0.05 … 0.75 applied as a mask over that one decoded set (`det_metrics.sweep_tau_boxes`).

**Both checkpoints get calibrated, and `last.pt` is the headline.** `train.py` writes `best.pt` on lowest *validation loss*, and on this run that selector stopped at an early epoch while AP, AP50, AP75 and F1 all kept improving to the last one — so the head anyone would actually ship is `last.pt`. Its optimal `tau` need not equal `best.pt`'s either: the score distribution keeps moving as the cosine anneals and the geometry branch converges. Each checkpoint is therefore swept on the same grid and reported **at its own `tau`**, never at one borrowed threshold.

Best `tau` = **argmin count MAE**. The F1-argmax `tau` is printed beside it as the sanity check: the best localisation should also give the lowest count error, and where the two disagree the report has to say so rather than quote whichever flatters.

**AP / AP50 / AP75 / AR100 do not move.** They integrate over the score axis, so they are threshold-independent by construction — calibration cannot change them, and the sweep does not pretend to report them. Only the operating-point numbers (P/R/F1, counts) respond to `tau`.

Then the NMS comparison, each checkpoint at its own calibrated `tau`: `box_nms_iou` None vs 0.5 vs 0.6 on the headline `last.pt`, None vs 0.5 on `best.pt` (enough to show the decision does not flip between them). Suppression happens **inside** the decode, before any thresholding, because NMS ranks by score and has to see the whole candidate set — so each setting costs its own pass (the `None` arm is the sweep above, already paid for).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display
from cropcounter.train import build_loaders, load_checkpoint
from cropcounter.det_metrics import sweep_tau_boxes

RUN_DIR  = f'{DRIVE}/runs/brackish_frozen_s0'
DOCS_IMG = f'{REPO}/examples/FishDetection/notebooks/docs/images'
os.makedirs(DOCS_IMG, exist_ok=True)

# BOTH checkpoints are calibrated. best.pt is whatever the val-loss selector picked;
# on this run that was an early epoch while every detection metric kept improving to
# the last one, so last.pt is the head anyone would ship -- and its optimal tau need
# not equal best.pt's, because the score distribution keeps shifting as the cosine
# anneals and the geometry branch converges. Same grid, one sweep pass each.
CKPTS = ('best', 'last')
LABEL = {'best': 'best epoch (val-loss)', 'last': 'last epoch'}
# The run's own config travels inside the checkpoint. Read the decode params off
# THAT, not off the config loaded in § 2 -- they must match the ones the targets
# were rendered with, and only the checkpoint knows what the run actually used.
models = {}
for ck in CKPTS:
    models[ck], tune_cfg = load_checkpoint(f'{RUN_DIR}/{ck}.pt', device, weights_dir='weights')
    models[ck].eval()
tune_cfg.data_root = pathlib.Path(DATA)
_, val_loader, _, val_recs = build_loaders(tune_cfg, device)
N_GT, N_IMG = sum(len(r.boxes) for r in val_recs), len(val_recs)
DEC = dict(ap_tau=tune_cfg.ap_tau, k=tune_cfg.k, top_k=tune_cfg.top_k,
           match_iou=tune_cfg.match_iou, output_stride=tune_cfg.output_stride,
           size_parameterisation=tune_cfg.size_parameterisation)
print(f'{N_IMG} val images | {N_GT} GT boxes | decode {DEC}')

TAUS = np.round(np.arange(0.05, 0.80, 0.05), 2)   # 0.05 -> 0.75 inclusive
calib = {}
for ck in CKPTS:
    # One forward pass, one decode at ap_tau, every tau a mask over that same set.
    # box_nms_iou is the run's own (None) -- i.e. this is also the NMS-off arm below.
    rows = sweep_tau_boxes(models[ck], val_loader, device, TAUS,
                           box_nms_iou=tune_cfg.box_nms_iou, progress=True, **DEC)
    mae_arg = min(rows, key=lambda r: r['count_mae'])
    f1_arg  = max(rows, key=lambda r: r['f1'])
    calib[ck] = {'best_tau': float(mae_arg['tau']),
                 'f1_argmax_tau': float(f1_arg['tau']), 'sweep': rows}
    print(f"\n--- {LABEL[ck]} ({ck}.pt) ---")
    for r in rows:
        print(f"tau {r['tau']:.2f} | MAE {r['count_mae']:7.3f} bias {r['count_bias']:+7.3f} | "
              f"P {r['precision']:.4f} R {r['recall']:.4f} F1 {r['f1']:.4f}")
    if abs(f1_arg['tau'] - mae_arg['tau']) < 1e-9:
        print(f"MAE-argmin and F1-argmax agree at tau {mae_arg['tau']:.2f}: best localisation "
              f"is also the lowest count error, as it should be.")
    else:
        print(f"MAE-argmin tau {mae_arg['tau']:.2f} (F1 {mae_arg['f1']:.4f}) DISAGREES with "
              f"F1-argmax tau {f1_arg['tau']:.2f} (F1 {f1_arg['f1']:.4f}) -- compensating "
              "errors. Report both; do not quote the flattering one.")

TAU = {ck: calib[ck]['best_tau'] for ck in CKPTS}
HEADLINE = 'last'
BEST_TAU = TAU[HEADLINE]   # the tau the results table below reads back

fig, axs = plt.subplots(1, 2, figsize=(17, 4.6))
for ax1, ck in zip(axs, CKPTS):
    rows = calib[ck]['sweep']; b = min(rows, key=lambda r: r['count_mae'])
    ax1.plot([r['tau'] for r in rows], [r['count_mae'] for r in rows], 'o-', color='#c0392b')
    ax1.set_xlabel('tau'); ax1.set_ylabel('count MAE', color='#c0392b')
    ax2 = ax1.twinx()
    ax2.plot([r['tau'] for r in rows], [r['f1'] for r in rows], 's-', color='#2b5f9e')
    ax2.set_ylabel('localisation F1', color='#2b5f9e')
    ax1.axvline(b['tau'], color='gray', ls='--', lw=1)
    ax1.set_title(f"best tau {b['tau']:.2f}: MAE {b['count_mae']:.2f}, "
                  f"F1 {b['f1']:.3f}, bias {b['count_bias']:+.2f}")
    # Which panel is which rides above the title, so the title string itself stays
    # byte-for-byte the format the wheat report's Figure 4 uses.
    ax1.text(0.5, 1.17, f'{LABEL[ck]} — {ck}.pt', transform=ax1.transAxes,
             ha='center', fontsize=11, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{DRIVE}/results/viz/tau_sweep.png', dpi=110, bbox_inches='tight')
shutil.copy(f'{DRIVE}/results/viz/tau_sweep.png', f'{DOCS_IMG}/tau_sweep.png')

agree = abs(TAU['last'] - TAU['best']) < 1e-9
print(f"\nVERDICT: best.pt calibrates to tau {TAU['best']:.2f}, last.pt to tau {TAU['last']:.2f} — "
      f"{'they agree' if agree else 'they DISAGREE, so each is scored at its own tau'}; headline = "
      f"last.pt, because the val-loss selector that wrote best.pt stopped at an earlier epoch "
      f"while AP / AP50 / AP75 / F1 all kept improving to the last.")
display(IPImage(f'{DRIVE}/results/viz/tau_sweep.png'))

In [ ]:
import matplotlib.patches
from collections import defaultdict
from PIL import Image
from cropcounter import autocast_context
from cropcounter.boxmap import clip_boxes_xyxy, decode_boxes

# NMS lives inside the decode, so each setting is its own pass, and each checkpoint
# is compared at ITS OWN calibrated tau. The None arm is the sweep row above --
# already paid for, not recomputed. The headline (last.pt) gets the full
# None/0.5/0.6 grid; best.pt only needs off vs 0.5 to show the decision does not
# flip between the two checkpoints.
NMS_GRID = {'last': ('none', '0.5', '0.6'), 'best': ('none', '0.5')}
nms_rows = {}
for ck, grid in NMS_GRID.items():
    nms_rows[ck] = {'none': dict(min(calib[ck]['sweep'], key=lambda r: r['count_mae']))}
    for label in grid:
        if label == 'none':
            continue
        nms_rows[ck][label] = sweep_tau_boxes(models[ck], val_loader, device, [TAU[ck]],
                                              box_nms_iou=float(label), progress=True, **DEC)[0]

# summarise_boxes reports rates, not raw counts. n_pred is pinned by the bias:
# bias = mean(n_pred - n_gt)  =>  sum(n_pred) = bias * n_images + sum(n_gt).
def n_pred_of(r):
    return round(r['count_bias'] * r['n_images'] + N_GT)

print('| Checkpoint | tau | NMS IoU | P | R | F1 | count MAE | n_pred |')
print('|' + '---|' * 8)
for ck in ('last', 'best'):
    for label in NMS_GRID[ck]:
        r = nms_rows[ck][label]
        print(f"| {LABEL[ck]} | {TAU[ck]:.2f} | {label} | {r['precision']:.4f} | "
              f"{r['recall']:.4f} | {r['f1']:.4f} | {r['count_mae']:.3f} | {n_pred_of(r)} |")
print(f"\neach checkpoint at its own calibrated tau | IoU {tune_cfg.match_iou} | "
      f"top-k {tune_cfg.top_k} | {N_GT} GT boxes over {N_IMG} val images")

# Spot check on the HEADLINE checkpoint: 4 val frames carrying >=3 GT fish,
# NMS off (left) vs NMS 0.5 (right), at last.pt's own calibrated tau.
gts_by_id = defaultdict(list)
for a in gt['annotations']:
    gts_by_id[a['image_id']].append(a['bbox'])
name_by_id = {im['id']: os.path.basename(im['file_name']) for im in gt['images']}
idx_by_id  = {r.image_id: i for i, r in enumerate(val_recs)}
# CFD filenames run ~90 chars (clip range + roboflow hash) and collide as titles.
def short(i): return name_by_id[i].split('_jpg.rf.')[0][-30:]
by_gt = sorted((i for i in name_by_id if i in idx_by_id), key=lambda i: -len(gts_by_id[i]))
picks = [i for i in by_gt if len(gts_by_id[i]) >= 3][:4]
picks += [i for i in by_gt if i not in picks][:4 - len(picks)]   # top up if the split is sparse

def decode_at(model, item, box_nms_iou, tau):
    """Decode one val item exactly as evaluate_boxes does, then keep scores > tau."""
    with torch.no_grad(), autocast_context(device):
        out = model(item['image'].unsqueeze(0).to(device))
    out = {n: v.float().cpu() for n, v in out.items()}
    boxes, scores = decode_boxes(
        torch.sigmoid(out['heatmap']), out['wh'], out['off'], stride=tune_cfg.output_stride,
        k=tune_cfg.k, tau=tune_cfg.ap_tau, top_k=tune_cfg.top_k, box_nms_iou=box_nms_iou,
        size_parameterisation=tune_cfg.size_parameterisation)
    return clip_boxes_xyxy(boxes, int(item['width']), int(item['height']))[scores > tau]

val_ds = val_loader.dataset
if not picks:
    print('no val frame carries a GT box — skipping the NMS spot-check figure')
else:
    fig2 = plt.figure(figsize=(16, 4.6 * len(picks)))
    axes = np.atleast_2d(fig2.subplots(len(picks), 2))
    for row, iid in enumerate(picks):
        item = val_ds[idx_by_id[iid]]
        img = Image.open(f'{DATA}/val/images/{name_by_id[iid]}')
        for col, iou in enumerate((None, 0.5)):
            ax = axes[row][col]; ax.imshow(img); ax.axis('off')
            pred = decode_at(models[HEADLINE], item, iou, TAU[HEADLINE])
            for x, y, w, h in gts_by_id[iid]:
                ax.add_patch(matplotlib.patches.Rectangle((x, y), w, h, fill=False,
                                                          edgecolor='lime', linewidth=1.2))
            for x0, y0, x1, y1 in pred:
                ax.add_patch(matplotlib.patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                                          edgecolor='red', linewidth=1.2))
            ax.set_title(f"…{short(iid)}  NMS {iou if iou else 'off'} · "
                         f"gt {len(gts_by_id[iid])} · ours {len(pred)}", fontsize=9)
    fig2.suptitle(f'{LABEL[HEADLINE]} ({HEADLINE}.pt) · GT green · ours red, '
                  f'at tau {TAU[HEADLINE]:.2f}', fontsize=11)
    fig2.savefig(f'{DRIVE}/results/viz/nms_compare.png', dpi=110, bbox_inches='tight')
    shutil.copy(f'{DRIVE}/results/viz/nms_compare.png', f'{DOCS_IMG}/nms_compare.png')
    display(IPImage(f'{DRIVE}/results/viz/nms_compare.png'))

In [ ]:
from cropcounter.det_metrics import evaluate_boxes, write_coco_results

# Table 2 analogue. "best epoch" means LOWEST VALIDATION LOSS -- the rule train.py
# writes best.pt on -- and on this run that selector disagreed with every detection
# metric, so both checkpoints are tabulated and the last epoch is the headline.
# Every row is reported at the tau ITS OWN checkpoint calibrated to; there is no
# single global tau in this table, and the footnote says so.
table = [(LABEL[ck], f"NMS {'off' if label == 'none' else label} @τ={TAU[ck]:.2f}", nms_rows[ck][label])
         for ck in ('best', 'last') for label in ('none', '0.5')]

lines = ['| Model | Config | MAE | RMSE | Bias | Precision | Recall | F1 |', '|' + '---|' * 8]
for model_name, config_name, r in table:
    lines.append(f"| {model_name} | {config_name} | {r['count_mae']:.3f} | {r['count_rmse']:.3f} | "
                 f"{r['count_bias']:+.3f} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1']:.4f} |")
lines += ['', 'Per-row tau, not one global tau: ' +
          ' · '.join(f'{LABEL[ck]} τ={TAU[ck]:.2f}' for ck in ('best', 'last')) +
          f' | IoU {tune_cfg.match_iou} | top-k {tune_cfg.top_k}', '',
          'Best epoch = lowest validation loss (the rule `train.py` writes `best.pt` on), not best '
          'AP. Headline checkpoint = **last epoch**: the val-loss selector stopped early while '
          'AP / AP50 / AP75 / F1 all kept improving.']
md = '\n'.join(lines)
print(md)
with open(f'{DRIVE}/results/tuning_table.md', 'w') as fh:
    fh.write(md + '\n')

# train.py only writes predictions.json for the best-val-loss epoch, so the last
# epoch has none on Drive. Decode one here -- same ap_tau, top-k, NMS setting and
# clipping evaluate_boxes used for best.pt -- so the results table can rescore the
# last epoch through the shared coco_eval and report its operating point at ITS tau.
_, _, last_dets = evaluate_boxes(models['last'], val_loader, device,
                                 gt=f'{DATA}/val/annotations.json', tau=TAU['last'],
                                 box_nms_iou=tune_cfg.box_nms_iou,
                                 progress=True, desc='last.pt detections', **DEC)
write_coco_results(last_dets, f'{RUN_DIR}/predictions_last.json')
print(f"\nwrote {RUN_DIR}/predictions_last.json ({len(last_dets)} detections)")

# The calibration the results table below reads back. Top-level best_tau is the
# HEADLINE checkpoint's, so the table and any older reader keep working unchanged;
# best.pt's own tau is best_tau_bestckpt, and the full per-checkpoint sweeps sit
# under 'best' / 'last'.
with open(f'{DRIVE}/results/tau_calibration.json', 'w') as fh:
    json.dump({'best_tau': TAU[HEADLINE], 'f1_argmax_tau': calib[HEADLINE]['f1_argmax_tau'],
               'best_tau_bestckpt': TAU['best'], 'headline_checkpoint': HEADLINE,
               'best': calib['best'], 'last': calib['last'], 'nms': nms_rows}, fh, indent=2)
print(f'wrote {DRIVE}/results/tau_calibration.json and {DRIVE}/results/tuning_table.md')

## 5 · Results table — ours vs baseline, same images, same scorer

In [ ]:
import csv
import numpy as np
from collections import defaultdict
from cropcounter.det_metrics import coco_eval, read_coco_results, sweep_tau_from_detections
from cropcounter.boxmap import boxes_xywh_to_xyxy
from cropcounter.dinov3_pyramid import PyramidDecoder

# The operating-point columns are reported at the CALIBRATED tau from § 4, read
# back from results/tau_calibration.json -- never the config's untuned 0.3. § 4
# calibrates each checkpoint separately, so tau varies BY ROW and is printed as
# its own column rather than baked into the headers.
# The AP columns are deliberately untouched: AP/AP50/AP75/AR100 integrate over
# the score axis, so they are threshold-independent and calibration cannot move
# them. Only P/R/F1 and the counts are recomputed here.
calib = json.load(open(f'{DRIVE}/results/tau_calibration.json'))
HEADLINE_TAU = float(calib['best_tau'])                                 # last.pt — the headline
BESTCKPT_TAU = float(calib.get('best_tau_bestckpt', HEADLINE_TAU))      # best.pt's own
CALIB_RUN = 'brackish_frozen_s0'   # the run § 4 calibrated; the probes borrow its headline tau
TAU_COL, F1_COL = 'τ', 'F1 @IoU.5 (τ per row, see Tuning)'
P_COL, R_COL, MAE_COL = 'precision @τ', 'recall @τ', 'count_mae @τ'

gt_boxes_by_id = defaultdict(list)
for a in gt['annotations']:
    gt_boxes_by_id[a['image_id']].append(a['bbox'])

def operating_point(predictions_path, tau):
    """P/R/F1 + count MAE at a calibrated tau, from the saved detections alone.

    No GPU and no model: predictions.json was decoded at ap_tau=0.01, so every
    operating threshold above it is a mask over detections we already have --
    the same pure helper sweep_tau_boxes is built on.
    """
    by_id = defaultdict(list)
    for d in read_coco_results(predictions_path):
        by_id[d['image_id']].append((d['bbox'], d['score']))
    per_image = []
    for im in gt['images']:
        entries = by_id.get(im['id'], [])
        per_image.append((
            boxes_xywh_to_xyxy(np.array([b for b, _ in entries], np.float32)),
            np.array([s for _, s in entries], np.float32),
            boxes_xywh_to_xyxy(np.array(gt_boxes_by_id[im['id']], np.float32)),
        ))
    return sweep_tau_from_detections(per_image, [tau], match_iou=cfg.match_iou)[0]

rows = []
for run in ('brackish_frozen_s0', 'brackish_linearprobe_s0', 'brackish_linearprobe_wheatinit_s0'):
    rd = f'{DRIVE}/runs/{run}'
    if not os.path.exists(f'{rd}/history.json'):
        continue
    h = json.load(open(f'{rd}/history.json'))
    best_ep = min(range(len(h['val_loss'])), key=lambda i: h['val_loss'][i])
    row = {'model': run, 'input': '1024 long side', 'epoch': best_ep + 1}
    for k in ('val_ap', 'val_ap50', 'val_ap75', 'val_ar100'):
        row[k.replace('val_', '')] = round(h[k][best_ep], 4) if k in h else None
    # Re-score the saved predictions with the same function used for the baseline — the parity check.
    tau_row = BESTCKPT_TAU if run == CALIB_RUN else HEADLINE_TAU
    if os.path.exists(f'{rd}/predictions.json'):
        row['ap_rescored'] = round(coco_eval(f'{DATA}/val/annotations.json', read_coco_results(f'{rd}/predictions.json'))['ap'], 4)
        op = operating_point(f'{rd}/predictions.json', tau_row)
        row[TAU_COL] = round(tau_row, 2); row[F1_COL] = round(op['f1'], 4)
        row[P_COL] = round(op['precision'], 4); row[R_COL] = round(op['recall'], 4)
        row[MAE_COL] = round(op['count_mae'], 3)
    rows.append(row)
    last_ep = len(h['val_loss']) - 1
    if last_ep != best_ep:
        # Best-by-val-loss is the house convention (Liam's examples); the last epoch is
        # reported beside it because on Brackish the val loss bottomed early while every
        # detection metric kept improving — which is why § 4 makes last.pt the headline.
        # § 4 also writes predictions_last.json for the calibrated run, so that row gets
        # a rescored AP and an operating point at the LAST checkpoint's own tau.
        last_row = {'model': f'{run} · last epoch', 'input': '1024 long side', 'epoch': last_ep + 1}
        for k in ('val_ap', 'val_ap50', 'val_ap75', 'val_ar100'):
            last_row[k.replace('val_', '')] = round(h[k][last_ep], 4) if k in h else None
        if os.path.exists(f'{rd}/predictions_last.json'):
            last_row['ap_rescored'] = round(coco_eval(f'{DATA}/val/annotations.json', read_coco_results(f'{rd}/predictions_last.json'))['ap'], 4)
            op = operating_point(f'{rd}/predictions_last.json', HEADLINE_TAU)
            last_row[TAU_COL] = round(HEADLINE_TAU, 2); last_row[F1_COL] = round(op['f1'], 4)
            last_row[P_COL] = round(op['precision'], 4); last_row[R_COL] = round(op['recall'], 4)
            last_row[MAE_COL] = round(op['count_mae'], 3)
        rows.append(last_row)
for name, m in baseline_metrics.items():
    rows.append({'model': f'{name} (released, likely saw these images)', 'input': name.split('_')[-1],
                 'gflops': m.get('gflops'),
                 **{k: round(m[k], 4) for k in ('ap', 'ap50', 'ap75', 'ar100')}})
# Parameter accounting — trainable AND total (the frozen ~89M backbone runs on every forward).
n_backbone = sum(p.numel() for p in box_model.backbone.parameters())
n_dec_box = sum(p.numel() for p in PyramidDecoder((128, 256, 512, 1024), task='box').parameters())
for r in rows:
    if r['model'].startswith('brackish'):
        r['trainable_params_M'] = round(n_dec_box / 1e6, 2); r['total_params_M'] = round((n_backbone + n_dec_box) / 1e6, 1)
        r['gflops'] = OURS_GFLOPS
keys = sorted({k for r in rows for k in r}, key=lambda k: (k != 'model', k))
with open(f'{DRIVE}/results/results.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(rows)
print('| ' + ' | '.join(keys) + ' |'); print('|' + '---|' * len(keys))
for r in rows:
    print('| ' + ' | '.join(str(r.get(k, '')) for k in keys) + ' |')

## 6 · Visual spot checks — GT (green) · ours (red) · RF-DETR (blue), six val images spanning the count range

In [ ]:
import numpy as np, matplotlib, matplotlib.patches
from PIL import Image
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from collections import defaultdict
ours = defaultdict(list); theirs = defaultdict(list); gts = defaultdict(list)
for d in read_coco_results(f'{DRIVE}/runs/brackish_frozen_s0/predictions.json'):
    if d['score'] >= 0.3: ours[d['image_id']].append(d['bbox'])
for d in read_coco_results(f'{DRIVE}/results/rfdetr_nano_640_predictions.json'):
    if d['score'] >= 0.3: theirs[d['image_id']].append(d['bbox'])
for a in gt['annotations']:
    gts[a['image_id']].append(a['bbox'])
name_by_id = {im['id']: os.path.basename(im['file_name']) for im in gt['images']}
by_count = sorted(name_by_id, key=lambda i: len(gts[i]))
picks = [by_count[int(q * (len(by_count) - 1))] for q in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)]
fig = Figure(figsize=(18, 12)); FigureCanvasAgg(fig)
for ax, iid in zip(fig.subplots(2, 3).ravel(), picks):
    ax.imshow(Image.open(f"{DATA}/val/images/{name_by_id[iid]}"))
    for boxes, col in ((gts[iid], 'lime'), (ours[iid], 'red'), (theirs[iid], 'deepskyblue')):
        for x, y, w, h in boxes:
            ax.add_patch(matplotlib.patches.Rectangle((x, y), w, h, fill=False, edgecolor=col, linewidth=1.2))
    ax.set_title(f"{name_by_id[iid]}  gt {len(gts[iid])} · ours {len(ours[iid])} · rfdetr {len(theirs[iid])}", fontsize=9); ax.axis('off')
fig.savefig(f'{DRIVE}/results/viz/spot_checks.png', dpi=110, bbox_inches='tight')
# Also drop the figures the report embeds into the repo checkout, to be committed from there.
DOCS_IMG = f'{REPO}/examples/FishDetection/notebooks/docs/images'
os.makedirs(DOCS_IMG, exist_ok=True)
shutil.copy(f'{DRIVE}/results/viz/spot_checks.png', f'{DOCS_IMG}/spot_checks.png')
for run in ('brackish_frozen_s0', 'brackish_linearprobe_s0'):
    if os.path.exists(f'{DRIVE}/runs/{run}/curves.png'):
        shutil.copy(f'{DRIVE}/runs/{run}/curves.png', f'{DOCS_IMG}/{run}_curves.png')
print('figures written to', DOCS_IMG)
from IPython.display import Image as IPImage, display
display(IPImage(f'{DRIVE}/results/viz/spot_checks.png'))

## 7 · Hand-off

Everything the memo needs is now on Drive: `results/results.csv`, `results/baseline_metrics.json`, `results/manifest/`, `results/throughput_probe.json`, `results/tau_calibration.json`, `results/tuning_table.md`, `results/viz/{spot_checks,tau_sweep,nms_compare}.png`, and the run dirs (each with `predictions.json`, plus `predictions_last.json` for the calibrated run). The figures the report embeds are also written into `docs/images/` in the repo checkout — commit them from there. Paste the tables above into [`docs/report.md`](docs/report.md) § Results and § Tuning, and apply the kill criterion: frozen head AP50 < ~0.5 while RF-DETR-Nano > ~0.8 ⇒ the frozen-trunk premise fails underwater → rescope to partial unfreeze or the label-efficiency claim alone.